In [53]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd


In [54]:
def series_has_team(series_data: dict, team_name="NRG") -> bool:
    team_name = team_name.lower()

    for game in series_data.get("seriesState", {}).get("games", []):
        for segment in game.get("segments", []):
            for team in segment.get("teams", []):
                name = team.get("name", "").lower()
                if team_name in name:
                    return True
    return False


In [55]:
from datetime import datetime

def extract_series_time(series_data: dict) -> datetime:
    for game in series_data.get("seriesState", {}).get("games", []):
        started = game.get("startedAt")
        if started:
            return datetime.fromisoformat(started.replace("Z", "+00:00"))
    return datetime.min


In [56]:
import json
from pathlib import Path

def get_all_nrg_series(base_dir="."):
    results = []

    for path in Path(base_dir).glob("series_*_raw.json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if series_has_team(data, "NRG"):
            results.append({
                "series_id": path.stem.replace("series_", "").replace("_raw", ""),
                "time": extract_series_time(data),
                "raw": data
            })

    return results


In [57]:
def get_last_5_nrg_matches():
    series = get_all_nrg_series(".")

    series.sort(key=lambda x: x["time"], reverse=True)
    return series[:5]


In [58]:
nrg_last_5 = get_last_5_nrg_matches()

print(f"Found {len(nrg_last_5)} NRG matches\n")

# for i, s in enumerate(nrg_last_5, 1):
#     print(f"{i}. Series {s['series_id']} | Time: {s['time']}")


Found 5 NRG matches



#stage1 feature extraction

In [59]:
import isodate

def avg_round_duration(series_data):
    durations = []

    for game in series_data["seriesState"]["games"]:
        for seg in game["segments"]:
            if seg["type"] == "round" and seg.get("finished"):
                dur = isodate.parse_duration(seg["duration"]).total_seconds()
                durations.append(dur)

    return sum(durations) / len(durations) if durations else 0


In [60]:
from datetime import timedelta

UTILITY_KEYWORDS = [
    "snake-bite", "paint-shells", "guided-salvo",
    "shock-dart", "grenade", "molotov"
]

def early_utility_rate(series_data):
    rounds_with_early_utility = 0
    total_rounds = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            total_rounds += 1
            round_start = datetime.fromisoformat(
                round_seg["startedAt"].replace("Z", "+00:00")
            )

            early_window = round_start + timedelta(seconds=15)
            found = False

            for team in round_seg.get("teams", []):
                for src in team.get("damageDealtSources", []):
                    weapon = src["source"]["name"].lower()
                    if any(u in weapon for u in UTILITY_KEYWORDS):
                        found = True
                        break

                if found:
                    break

            if found:
                rounds_with_early_utility += 1

    return rounds_with_early_utility / total_rounds if total_rounds else 0


In [61]:
def avg_first_contact_time(series_data):
    timings = []

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            elapsed = 0
            found = False

            for team in round_seg.get("teams", []):
                if team.get("damageDealt", 0) > 0:
                    found = True
                    break

            if found:
                # conservative proxy: assume first damage at 15–25s
                timings.append(20)

    return sum(timings) / len(timings) if timings else 0


In [62]:
def site_hit_frequency(series_data):
    hits = 0
    attack_rounds = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            for team in round_seg.get("teams", []):
                if team.get("side") == "attacker":
                    attack_rounds += 1
                    if any(obj["type"] == "plantBomb" for obj in team.get("objectives", [])):
                        hits += 1
                    break

    return hits / attack_rounds if attack_rounds else 0


In [63]:
def retake_vs_hold_rate(series_data):
    retake = 0
    hold = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            plant_occurred = any(
                obj["type"] == "plantBomb"
                for t in round_seg.get("teams", [])
                for obj in t.get("objectives", [])
            )

            for team in round_seg.get("teams", []):
                if team.get("side") == "defender" and team.get("won"):
                    if plant_occurred:
                        retake += 1
                    else:
                        hold += 1

    total = retake + hold
    return {
        "retake_rate": retake / total if total else 0,
        "hold_rate": hold / total if total else 0
    }


In [64]:
# features = []

# for s in nrg_last_5:
#     raw = s["raw"]
#     features.append({
#         "series_id": s["series_id"],
#         "avg_round_duration": avg_round_duration(raw),
#         "early_utility_rate": early_utility_rate(raw),
#         "first_contact_time": avg_first_contact_time(raw),
#         "site_hit_freq": site_hit_frequency(raw),
#         **retake_vs_hold_rate(raw)
#     })

# import pandas as pd
# df = pd.DataFrame(features)
# df


In [65]:
import isodate
from datetime import datetime

UTILITY_KEYWORDS = [
    "snake-bite", "paint-shells", "guided-salvo",
    "shock-dart", "grenade", "molotov"
]

def iter_rounds(series_data):
    """
    Generator yielding finished round segments
    """
    for game in series_data.get("seriesState", {}).get("games", []):
        for seg in game.get("segments", []):
            if seg.get("type") == "round" and seg.get("finished"):
                yield seg


In [66]:
def pistol_conversion_rate(series_data):
    rounds = list(iter_rounds(series_data))
    pistol_indices = [0, 12]  # standard halves

    conversions = 0
    total = 0

    for idx in pistol_indices:
        if idx + 1 >= len(rounds):
            continue

        pistol = rounds[idx]
        follow = rounds[idx + 1]

        for team in pistol["teams"]:
            if team.get("won"):
                total += 1
                follow_team = next(
                    t for t in follow["teams"] if t["id"] == team["id"]
                )
                if follow_team.get("won"):
                    conversions += 1

    return conversions / total if total else 0


In [67]:
def mid_round_damage_ratio(series_data):
    ratios = []

    for r in iter_rounds(series_data):
        total_damage = 0
        mid_damage = 0

        for team in r["teams"]:
            total_damage += team.get("damageDealt", 0)

            # proxy: assume mid-round utility contributes here
            for src in team.get("damageDealtSources", []):
                total_damage += src["damageAmount"]
                if any(u in src["source"]["name"].lower() for u in UTILITY_KEYWORDS):
                    mid_damage += src["damageAmount"]

        if total_damage > 0:
            ratios.append(mid_damage / total_damage)

    return sum(ratios) / len(ratios) if ratios else 0


In [68]:
def avg_default_duration(series_data):
    durations = []

    for r in iter_rounds(series_data):
        start = datetime.fromisoformat(r["startedAt"].replace("Z", "+00:00"))

        for team in r["teams"]:
            if team.get("objectives") or team.get("damageDealtSources"):
                # proxy default duration
                durations.append(20)
                break

    return sum(durations) / len(durations) if durations else 0


In [69]:
def trade_efficiency(series_data):
    assists = 0
    kills = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            assists += team.get("killAssistsReceived", 0)
            kills += team.get("kills", 0)

    return assists / kills if kills else 0


In [70]:
def first_blood_participation(series_data):
    supported = 0
    total = 0

    for r in iter_rounds(series_data):
        total += 1
        contributors = sum(
            1 for team in r["teams"] if team.get("damageDealt", 0) > 0
        )
        if contributors >= 2:
            supported += 1

    return supported / total if total else 0


In [71]:
def late_round_win_rate(series_data):
    wins = 0
    total = 0

    for r in iter_rounds(series_data):
        dur = isodate.parse_duration(r["duration"]).total_seconds()

        if dur > 70:
            total += 1
            if any(t.get("won") for t in r["teams"]):
                wins += 1

    return wins / total if total else 0


In [72]:
def utility_damage_share(series_data):
    utility = 0
    total = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            total += team.get("damageDealt", 0)

            for src in team.get("damageDealtSources", []):
                dmg = src["damageAmount"]
                total += dmg
                if any(u in src["source"]["name"].lower() for u in UTILITY_KEYWORDS):
                    utility += dmg

    return utility / total if total else 0


In [73]:
def post_plant_success_rate(series_data):
    wins = 0
    plants = 0

    for r in iter_rounds(series_data):
        planted = any(
            obj["type"] == "plantBomb"
            for t in r["teams"]
            for obj in t.get("objectives", [])
        )

        if planted:
            plants += 1
            if any(t.get("won") for t in r["teams"]):
                wins += 1

    return wins / plants if plants else 0


In [74]:
def defensive_aggression_rate(series_data):
    early = 0
    total = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            if team.get("side") == "defender":
                total += 1
                if team.get("damageDealt", 0) > 0:
                    early += 1
                break

    return early / total if total else 0


In [75]:
def round_collapse_rate(series_data):
    collapse = 0
    total = 0

    for r in iter_rounds(series_data):
        first_kill_team = None

        for team in r["teams"]:
            if team.get("firstKill"):
                first_kill_team = team
                break

        if first_kill_team:
            total += 1
            if not first_kill_team.get("won"):
                collapse += 1

    return collapse / total if total else 0


In [77]:
features = []

for s in nrg_last_5:
    raw = s["raw"]
    features.append({
        "series_id": s["series_id"],
        "series_id": s["series_id"],
        "avg_round_duration": avg_round_duration(raw),
        "early_utility_rate": early_utility_rate(raw),
        "first_contact_time": avg_first_contact_time(raw),
        "site_hit_freq": site_hit_frequency(raw),
        **retake_vs_hold_rate(raw),
        "pistol_conv": pistol_conversion_rate(raw),
        "mid_round_ratio": mid_round_damage_ratio(raw),
        "default_duration": avg_default_duration(raw),
        "trade_eff": trade_efficiency(raw),
        "first_blood_support": first_blood_participation(raw),
        "late_round_win": late_round_win_rate(raw),
        "utility_share": utility_damage_share(raw),
        "post_plant": post_plant_success_rate(raw),
        "def_aggression": defensive_aggression_rate(raw),
        "collapse_rate": round_collapse_rate(raw),
    })


In [78]:
import pandas as pd
df = pd.DataFrame(features)
df


,series_id,avg_round_duration,early_utility_rate,first_contact_time,site_hit_freq,retake_rate,hold_rate,pistol_conv,mid_round_ratio,default_duration,trade_eff,first_blood_support,late_round_win,utility_share,post_plant,def_aggression,collapse_rate
0,2843071,141.477926,0.352941,20.0,0.602941,0.325000,0.675000,1.0,0.009412,20.0,0.420824,1.000000,1.0,0.009232,1.0,1.0,0.264706
1,2843070,156.255824,0.317647,20.0,0.788235,0.540541,0.459459,1.0,0.004036,20.0,0.463248,0.988235,1.0,0.004021,1.0,1.0,0.364706
2,2843069,145.983049,0.508197,20.0,0.704918,0.500000,0.500000,1.0,0.011975,20.0,0.464368,1.000000,1.0,0.011972,1.0,1.0,0.278689
3,2843067,148.336500,0.200000,20.0,0.533333,0.350000,0.650000,1.0,0.002139,20.0,0.320755,1.000000,1.0,0.002247,1.0,1.0,0.133333
4,2843063,132.850224,0.268657,20.0,0.626866,0.500000,0.500000,1.0,0.006008,20.0,0.403888,1.000000,1.0,0.005658,1.0,1.0,0.313433


In [79]:
df.to_csv("team_strategy_features.csv", index=False)
